# N118 · KMP与Z函数

**目标：** 利用已匹配前缀避免重复比较。

**先修：** N006, N013, N077。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 前缀函数；失配链接；Z区间；重叠匹配；总比较次数。

## 从问题到算法

朴素匹配失败后把文本指针退回，重复比较已经相等的前缀。KMP用前缀函数记录“当前匹配前缀的最长真前后缀”，失败时只回退模式长度。Z函数问的是每个位置与整个串前缀能匹配多长；二者都通过复用已知相等区间避免重做比较。

## 最小实现

**本章接口：** `prefix_function(s)`、`kmp_find(text, pattern)`、`z_function(s)`、`sum_scores(s)`

In [1]:
def prefix_function(s):
    pi=[0]*len(s)
    for i in range(1,len(s)):
        j=pi[i-1]
        while j and s[i]!=s[j]: j=pi[j-1]
        if s[i]==s[j]: j+=1
        pi[i]=j
    return pi

def kmp_find_all(text,pattern):
    if not pattern: return list(range(len(text)+1))
    pi=prefix_function(pattern); j=0; out=[]
    for i,c in enumerate(text):
        while j and c!=pattern[j]: j=pi[j-1]
        if c==pattern[j]: j+=1
        if j==len(pattern): out.append(i-j+1); j=pi[j-1]
    return out

def kmp_find(text,pattern):
    if not pattern: return 0
    pi=prefix_function(pattern); j=0
    for i,c in enumerate(text):
        while j and c!=pattern[j]: j=pi[j-1]
        if c==pattern[j]: j+=1
        if j==len(pattern): return i-j+1
    return -1

def z_function(s):
    z=[0]*len(s); left=right=0
    for i in range(1,len(s)):
        if i<right: z[i]=min(right-i,z[i-left])
        while i+z[i]<len(s) and s[z[i]]==s[i+z[i]]: z[i]+=1
        if i+z[i]>right: left,right=i,i+z[i]
    return z

def sum_scores(s): return len(s)+sum(z_function(s))

## 正确性、前提与复杂度

KMP状态j表示模式前缀等于已读文本的最长后缀，失配时只有它的真前后缀仍可能匹配。j每次成功至多增加1，失败只下降，摊还比较次数线性。Z盒内复用结果不能越过右边界，越界后才逐字符扩展；右边界总共前进O(n)次。

**代价：** 前缀函数/Z各O(n)时间空间；KMP O(|text|+|pattern|)时间、O(|pattern|)辅助（全部匹配另加输出空间）。约定Z[0]=0，总评分另加串长；空模式匹配所有n+1个边界。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='ababab'; pi=prefix_function(s); z=z_function(s)
show_table(['下标','字符','最长真前后缀','Z值'],[(i,c,pi[i],z[i]) for i,c in enumerate(s)])
print('重叠匹配位置',kmp_find_all(s,'abab'))

下标,字符,最长真前后缀,Z值
0,a,0,0
1,b,0,0
2,a,1,4
3,b,2,0
4,a,3,2
5,b,4,0


重叠匹配位置 [0, 2]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert kmp_find_all('ababab','abab')==[0,2]
assert kmp_find('abc','d')==-1 and kmp_find('abc','')==0
assert sum_scores('babab')==9
from itertools import product
words=[''.join(p) for n in range(6) for p in product('ab',repeat=n)]
for s in words:
    pi=prefix_function(s); z=z_function(s)
    for i in range(len(s)):
        assert pi[i]==max((k for k in range(1,i+1) if s[:k]==s[i-k+1:i+1]),default=0)
        if i: assert z[i]==max(k for k in range(len(s)-i+1) if s[:k]==s[i:i+k])
    for p in words[:15]:
        assert kmp_find(s,p)==s.find(p)
        assert kmp_find_all(s,p)==[i for i in range(len(s)+1) if s.startswith(p,i)]
print('N118: 所有本章断言通过')

N118: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 输出所有重叠匹配位置。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 构造多次回跳的周期字符串。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 28. Find the Index of the First Occurrence in a String（canonical）
- 459. Repeated Substring Pattern（canonical）
- 2223. Sum of Scores of Built Strings（canonical）：[官方入口](https://leetcode.com/problems/sum-of-scores-of-built-strings/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。